# Results: linear probes and domain transfer

Frozen SimCLR and NNCLR encoders evaluated with a linear probe on TNG, then applied zero-shot to Horizon-AGN and HSC. Produces Tables 4.1–4.7 and Figure 4.1 of the thesis. Clustering (Table 4.8, Figures 4.2–4.6) is in `clustering.ipynb`.

The latents come from `simclr_arm.py` / `nnclr_arm.py` (TNG) and `encode_stack.py` (Horizon, HSC). MB24 = Margalef-Bentabol et al. (2024).

## Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

TAG = "v3"
ROOT = Path("/scratch/s4743091/tng")
PROC = ROOT / "processed_tng"
MODELS = ROOT / "models"
RESULTS = ROOT / "results_notebook" / TAG
FIGS = RESULTS / "figs"
for d in (RESULTS, FIGS): d.mkdir(parents=True, exist_ok=True)

SPLIT_SEED, VAL_FRAC = 42, 0.2

ARMS = {
    "simclr": MODELS / "simclr_tng/latents_simclr.npy",
    "nnclr":  MODELS / "nnclr_tng/latents_nnclr.npy",
}

# Margalef-Bentabol+2024 Table 2 (TNG-test, all z) — reference rows
CHALLENGE = pd.DataFrame({
    "method":["RF","Swin","Zoobot","CNN1","CNN2","CNN3"],
    "acc":[70.5,72.6,77.7,74.1,70.6,71.5], "prec":[71.6,69.1,80.1,76.6,72.0,73.3],
    "rec":[68.1,81.9,73.8,69.5,67.2,67.6], "f1":[69.8,74.9,76.8,72.8,69.5,70.4],
    "auc":[77.0,81.2,85.1,80.9,76.5,76.9]}).set_index("method")

plt.rcParams.update({"figure.dpi":110, "savefig.dpi":200, "savefig.bbox":"tight"})
print("results →", RESULTS)

In [ ]:
def cache_array(name, fn, force=False):
    p = RESULTS / f"{name}.npy"
    if p.exists() and not force: return np.load(p)
    out = np.asarray(fn()); np.save(p, out)
    print(f"[cached] {p.name} {out.shape}"); return out

def cache_df(name, fn, force=False):
    p = RESULTS / f"{name}.csv"
    if p.exists() and not force: return pd.read_csv(p, index_col=0)
    out = fn(); out.to_csv(p); print(f"[cached] {p.name}"); return out

class cached_fig:
    def __init__(self, name, force=False):
        self.path = FIGS / f"{name}.png"; self.force = force
    def __enter__(self):
        self.run = self.force or not self.path.exists(); return self.run
    def __exit__(self, *exc):
        if exc[0] is not None: return False
        if self.run: plt.savefig(self.path); plt.close()
        display(Image(str(self.path))); return False

In [ ]:
lab = np.load(PROC / "labels_tng.npy")
meta = pd.DataFrame({n: lab[n] for n in lab.dtype.names})
meta["is_major_merger"] = meta["is_major_merger"].astype(int)
N = len(meta)
print(N, "galaxies | merger frac", meta.is_major_merger.mean().round(3))
print("redshift", meta.redshift.min(), "→", meta.redshift.max(), "| sequences", meta.sequence_ID.nunique())

## Merger-tree-disjoint split

Galaxies from the same merger tree (`sequence_ID`) never appear in both train and validation.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

tr_idx, va_idx = next(GroupShuffleSplit(1, test_size=VAL_FRAC, random_state=SPLIT_SEED)
                      .split(np.zeros(N), groups=meta.sequence_ID.values))
train_mask = np.zeros(N, bool); train_mask[tr_idx] = True
val_mask   = np.zeros(N, bool); val_mask[va_idx]  = True
assert not (set(meta.sequence_ID[train_mask]) & set(meta.sequence_ID[val_mask])), "sequence leakage!"
print(f"split: {train_mask.sum()} train / {val_mask.sum()} val")

for arm, path in ARMS.items():
    X = np.load(path, mmap_mode="r")
    assert X.shape[0] == N, f"{arm}: {X.shape[0]} rows != {N} labels — not the full-dataset array?"
    print(f"{arm:8s} latents {X.shape}")

## Linear probes

One logistic regression per arm on the standardised 512-d features; validation scores are cached to disk.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

y = meta.is_major_merger.values

def fit_arm(arm):
    X = np.load(ARMS[arm]).astype(np.float32)
    sc = StandardScaler().fit(X[train_mask])
    Xtr, Xva = sc.transform(X[train_mask]), sc.transform(X[val_mask])
    ytr = y[train_mask]
    p_lin = LogisticRegression(max_iter=2000).fit(Xtr, ytr).predict_proba(Xva)[:, 1]
    return p_lin

# val rows are identical across arms (same global split), so one shared index array
val_idx = np.where(val_mask)[0]
np.save(RESULTS / "val_idx.npy", val_idx)
y_val = y[val_idx]
z_val = meta.redshift.values[val_idx]

P = {}
for arm in ARMS:
    f_lin = RESULTS / f"p_val_{arm}_linear.npy"
    if not f_lin.exists():
        np.save(f_lin, fit_arm(arm))
        print(f"[fit+cached] {arm}")
    P[(arm, "linear")] = np.load(RESULTS / f"p_val_{arm}_linear.npy")

## Table 4.1: binary metrics vs MB24

The `Clust.` column of the thesis table comes from `clustering.ipynb`.

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

def row(p, thr=0.5):
    pred = (p >= thr).astype(int)
    return dict(acc=100*accuracy_score(y_val, pred), prec=100*precision_score(y_val, pred, zero_division=0),
                rec=100*recall_score(y_val, pred, zero_division=0), f1=100*f1_score(y_val, pred, zero_division=0),
                auc=100*roc_auc_score(y_val, p))

def build():
    ours = pd.DataFrame({f"{a} ({pr})": row(p) for (a, pr), p in P.items()}).T.round(1)
    ref = CHALLENGE.copy(); ref.index = [f"MB24 {m}" for m in ref.index]
    return pd.concat([ours, ref])

cache_df("metrics_table", build, force=True)

## Bootstrap uncertainty, per-redshift metrics (Table 4.3) and ROC (Figure 4.1)

These functions are from `probe_diagnostics.py`. The original read the evaluation arrays from `.npy` files; here they are taken directly from the probe cells above (`d` below). Its merger-tree leakage check is not repeated: the split cell already asserts that no `sequence_ID` is shared.

In [ ]:
from sklearn.metrics import roc_curve

d = dict(y_true=y_val, z=z_val,
         y_simclr=P[("simclr", "linear")], y_nnclr=P[("nnclr", "linear")])

# MB24 Table values (all-z, TNG-test): name: (precision%, recall%, AUC%)
MB24 = {
    "RF":     (71.6, 68.1, 77.0),
    "Swin":   (69.1, 81.9, 81.2),
    "Zoobot": (80.1, 73.8, 85.1),
    "CNN1":   (76.6, 69.5, 80.9),
    "CNN2":   (72.0, 67.2, 76.5),
    "CNN3":   (73.3, 67.6, 76.9),
}
Z_EDGES = [0.1, 0.31, 0.52, 0.76, 1.0]

In [ ]:
def bootstrap_auc(y_true, scores, n_boot=2000, seed=0):
    """Percentile bootstrap over the eval set. Returns (auc, lo, hi, std)."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    aucs = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        if y_true[idx].sum() in (0, n):
            aucs[b] = np.nan; continue
        aucs[b] = roc_auc_score(y_true[idx], scores[idx])
    aucs = aucs[~np.isnan(aucs)]
    return (roc_auc_score(y_true, scores),
            np.percentile(aucs, 2.5), np.percentile(aucs, 97.5), aucs.std())

def paired_auc_diff(y_true, s_a, s_b, n_boot=2000, seed=0):
    """Paired bootstrap of AUC(a) - AUC(b) on identical resampled rows."""
    rng = np.random.default_rng(seed)
    n = len(y_true); diffs = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        if y_true[idx].sum() in (0, n):
            diffs[b] = np.nan; continue
        diffs[b] = roc_auc_score(y_true[idx], s_a[idx]) - roc_auc_score(y_true[idx], s_b[idx])
    diffs = diffs[~np.isnan(diffs)]
    obs = roc_auc_score(y_true, s_a) - roc_auc_score(y_true, s_b)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    p_two = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return obs, lo, hi, p_two

def report_bootstrap(d):
    print("\n=== bootstrap AUC (95% CI over eval set) ===")
    for name, key in [("SimCLR", "y_simclr"), ("NNCLR", "y_nnclr")]:
        auc, lo, hi, sd = bootstrap_auc(d["y_true"], d[key])
        print(f"  {name:7s}: AUC = {auc:.3f}  95% CI [{lo:.3f}, {hi:.3f}]  (+/- {sd:.3f})")
    obs, lo, hi, p = paired_auc_diff(d["y_true"], d["y_simclr"], d["y_nnclr"])
    sig = "significant" if (lo > 0 or hi < 0) else "NOT significant"
    print(f"  SimCLR - NNCLR: {obs:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  "
          f"p~{p:.3f}  -> {sig}")

def per_bin_metrics(d, edges=Z_EDGES, thr=0.5):
    print("\n=== per-redshift-bin metrics ===")
    bins = np.digitize(d["z"], edges[1:-1])
    for name, key in [("SimCLR", "y_simclr"), ("NNCLR", "y_nnclr")]:
        print(f"\n  {name}")
        print(f"  {'z-bin':<14}{'N':>7}{'P':>7}{'R':>7}{'F1':>7}{'AUC':>7}")
        first = last = None
        for b in range(len(edges) - 1):
            m = bins == b
            yt, ys = d["y_true"][m], d[key][m]
            yp = (ys >= thr).astype(int)
            P_ = precision_score(yt, yp, zero_division=0)
            R_ = recall_score(yt, yp, zero_division=0)
            F1 = 0 if P_ + R_ == 0 else 2 * P_ * R_ / (P_ + R_)
            A = roc_auc_score(yt, ys) if 0 < yt.sum() < len(yt) else np.nan
            lab_ = f"[{edges[b]:.2f},{edges[b+1]:.2f})"
            print(f"  {lab_:<14}{m.sum():>7}{P_:>7.3f}{R_:>7.3f}{F1:>7.3f}{A:>7.3f}")
            if b == 0: first = (P_, R_)
            if b == len(edges) - 2: last = (P_, R_)
        dP, dR = 100*(first[0]-last[0]), 100*(first[1]-last[1])
        print(f"  drop bin1->bin4:  precision {dP:+.1f}%   recall {dR:+.1f}%   "
              f"(MB24: ~5% / ~9%)")

def plot_roc(d, out="roc_tng.pdf"):
    fig, ax = plt.subplots(figsize=(5.0, 5.0))
    ax.plot([0, 1], [0, 1], ls="--", lw=1, color="0.6", zorder=1)
    # MB24 operating points: TPR = recall, FPR = recall*(1/precision - 1) (balanced test set)
    for name, (P_, R_, A) in MB24.items():
        P_, R_ = P_/100, R_/100
        fpr, tpr = R_*(1/P_ - 1), R_
        ax.scatter(fpr, tpr, s=45, marker="o", facecolor="none",
                   edgecolor="0.45", lw=1.3, zorder=3)
        ax.annotate(f"{name}", (fpr, tpr), textcoords="offset points",
                    xytext=(6, -3), fontsize=7.5, color="0.35")
    for key, name, col in [("y_simclr", "SimCLR (this work)", "#d1495b"),
                           ("y_nnclr",  "NNCLR (this work)",  "#2e7d9c")]:
        fpr, tpr, _ = roc_curve(d["y_true"], d[key])
        auc = roc_auc_score(d["y_true"], d[key])
        ax.plot(fpr, tpr, lw=2.6, color=col, zorder=4,
                label=f"{name}  (AUC {auc:.3f})")
    ax.scatter([], [], s=45, marker="o", facecolor="none", edgecolor="0.45",
               lw=1.3, label="MB24 methods (operating points)")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_aspect("equal")
    ax.legend(loc="lower right", fontsize=8, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(out, bbox_inches="tight")
    print(f"\nwrote {out}")

In [ ]:
report_bootstrap(d)
per_bin_metrics(d)
plot_roc(d, out="roc_tng.pdf")

## Table 4.2: pairwise stage separation

For each pair of classes, a probe is trained and scored on galaxies of those two classes only.

In [ ]:
from itertools import combinations

B = "/scratch/s4743091/tng/results_notebook/v3/"
PR = "/scratch/s4743091/tng/processed_tng/"
M = {"SimCLR": "/scratch/s4743091/tng/models/simclr_tng/latents_simclr.npy",
     "NNCLR":  "/scratch/s4743091/tng/models/nnclr_tng/latents_nnclr.npy"}
CLASSES = ["non", "pre", "ongoing", "post"]

lab_s   = np.load(PR + "labels_tng_seqsplit.npy", allow_pickle=True)
val_idx_s = np.load(B + "val_idx.npy")
stage = np.select([lab_s["is_pre_merger"]==1, lab_s["is_ongoing_merger"]==1, lab_s["is_post_merger"]==1],
                  [1, 2, 3], default=0).astype(int)
va = np.zeros(len(stage), bool); va[val_idx_s] = True; tr = ~va

for arm, fpath in M.items():
    X = np.load(fpath).astype(np.float32)
    sc = StandardScaler().fit(X[tr]); Xa = sc.transform(X)
    print(f"\n=== {arm} : pairwise AUC (class A vs class B only) ===")
    for a, b in combinations(range(4), 2):
        mtr = tr & np.isin(stage, [a, b]);  mva = va & np.isin(stage, [a, b])
        ytr = (stage[mtr] == b).astype(int); yva = (stage[mva] == b).astype(int)
        clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xa[mtr], ytr)
        auc = roc_auc_score(yva, clf.predict_proba(Xa[mva])[:, 1])
        print(f"  {CLASSES[a]:>7} vs {CLASSES[b]:<7}  AUC = {auc:.3f}   "
              f"(val n: {int((stage[mva]==a).sum())} / {int((stage[mva]==b).sum())})")

## Table 4.4: redshift distribution of shared misclassifications

Galaxies that both probes misclassify at a 0.5 threshold.

In [ ]:
votes = np.stack([(P[(a, "linear")] >= 0.5).astype(int) for a in ARMS])
all_wrong = (votes != y_val).all(axis=0)
fp_local = np.where(all_wrong & (y_val == 0))[0]   # both arms say merger, label no
fn_local = np.where(all_wrong & (y_val == 1))[0]   # both arms say non-merger, label yes
fp, fn = val_idx[fp_local], val_idx[fn_local]       # → global rows
np.save(RESULTS / "consensus_fp_idx.npy", fp); np.save(RESULTS / "consensus_fn_idx.npy", fn)
print(f"consensus FP {len(fp)} | FN {len(fn)} of {len(y_val)} val")
display(meta.loc[np.r_[fp, fn]].assign(kind=["FP"]*len(fp)+["FN"]*len(fn))
            .groupby("kind")[["redshift"]].describe().round(2))

In [ ]:
edges  = [0.31, 0.52, 0.76]
labels = ["[0.10,0.31)", "[0.31,0.52)", "[0.52,0.76)", "[0.76,1.00)"]

for name, idx in [("FP", fp), ("FN", fn)]:
    z = meta.loc[idx, "redshift"].to_numpy()
    counts = np.bincount(np.digitize(z, edges), minlength=4)
    frac = np.round(counts / counts.sum(), 3)
    print(name, dict(zip(labels, frac)), "N =", int(counts.sum()))

---
## Domain transfer: TNG → Horizon-AGN, HSC

This section fits its own probe, on a random 60,000-galaxy sample of the TNG training rows, and applies it unchanged to every domain. The encoder, scaler and probe are never refit. This is the probe behind Tables 4.5–4.7, so its TNG AUC can differ from Table 4.1 in the third decimal.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, accuracy_score

# ---- per-domain config (edit paths; leave a path blank/missing to skip that domain) ----
SOURCE = "tng"
DOMAINS = {
    "tng": {
        "labels": "/scratch/s4743091/tng/processed_tng/labels_tng_seqsplit.npy",
        "simclr": "/scratch/s4743091/tng/models/simclr_tng/latents_simclr.npy",
        "nnclr":  "/scratch/s4743091/tng/models/nnclr_tng/latents_nnclr.npy",
    },
    "horizon": {
        "labels": "/scratch/s4743091/horizon/labels_horizon.npy",
        "simclr": "/scratch/s4743091/horizon/latents_simclr_horizon.npy",
        "nnclr":  "/scratch/s4743091/horizon/latents_nnclr_horizon.npy",
    },
    "hsc": {
        "labels": "/scratch/s4743091/hsc/labels_hsc.npy",
        "simclr": "/scratch/s4743091/hsc/latents_simclr_hsc.npy",
        "nnclr":  "/scratch/s4743091/hsc/latents_nnclr_hsc.npy",
    },
}
ARM_LIST = ["simclr", "nnclr"]
TRAIN_SAMPLE = 60000
RNG = np.random.default_rng(0)

# paper reference bands (Margalef+24): (auc_lo, auc_hi), (P_lo,P_hi), (R_lo,R_hi)
SUP = {
    "tng":     {"auc": (0.765, 0.851), "P": (0.69, 0.80), "R": (0.67, 0.82)},
    "horizon": {"auc": (0.549, 0.672), "P": (0.654, 0.72), "R": (0.124, 0.473)},
    "hsc":     {"auc": (0.740, 0.851), "P": (0.71, 0.85), "R": (0.58, 0.75)},
}

def load_labels(path):
    lab = np.load(path)
    nm = lab.dtype.names or ()
    y = lab["is_major_merger"].astype(int)
    sub = lab["subfolder"] if "subfolder" in nm else None
    seq = lab["sequence_ID"] if "sequence_ID" in nm else None
    dist = lab["is_disturbed"].astype(int) if "is_disturbed" in nm else None
    labeled = (y >= 0)                      # -99 sentinel = unlabeled (HSC)
    return y, sub, seq, dist, labeled

DATA = {}
print("availability:")
for dom, cfg in DOMAINS.items():
    if not os.path.exists(cfg["labels"]):
        print(f"  {dom:8s} labels MISSING ({cfg['labels']}) -> skip domain"); continue
    y, sub, seq, dist, labeled = load_labels(cfg["labels"])
    entry = {"y": y, "sub": sub, "seq": seq, "dist": dist,
             "labeled": labeled, "feat": {}}
    for m in ARM_LIST:
        p = cfg.get(m, "")
        ok = p and os.path.exists(p)
        if ok:
            arr = np.load(p, mmap_mode="r")
            if len(arr) != len(y):
                print(f"  {dom:8s} {m:6s} ROW MISMATCH {len(arr)} vs {len(y)} -> skip"); continue
            entry["feat"][m] = arr
        print(f"  {dom:8s} {m:6s} {'OK' if ok else 'MISSING':7s} "
              f"(n={len(y):,}, labeled={labeled.sum():,})")
    if entry["feat"]:
        DATA[dom] = entry
TARGETS = [d for d in DATA if d != SOURCE]
print("\nsource:", SOURCE, "| targets:", TARGETS, "| models:", [m for m in ARM_LIST])
assert SOURCE in DATA, "source domain features/labels missing -- nothing to transfer from"

In [ ]:
src = DATA[SOURCE]
tr_pool = np.where(src["sub"] == "train")[0] if src["sub"] is not None else np.arange(len(src["y"]))
TR = np.sort(RNG.choice(tr_pool, size=min(TRAIN_SAMPLE, len(tr_pool)), replace=False))

PROBE, SCAL = {}, {}
for m in src["feat"]:
    Xtr = np.asarray(src["feat"][m][TR], dtype=np.float32)
    sc = StandardScaler().fit(Xtr)
    lr = LogisticRegression(max_iter=2000).fit(sc.transform(Xtr), src["y"][TR])
    SCAL[m], PROBE[m] = sc, lr
    print(f"trained {m} source probe on {len(TR):,} TNG rows")

def metrics(y, p, thr=0.5):
    pred = (p >= thr).astype(int)
    return {"auc": roc_auc_score(y, p), "acc": accuracy_score(y, pred),
            "P": precision_score(y, pred, zero_division=0),
            "R": recall_score(y, pred, zero_division=0),
            "F1": f1_score(y, pred, zero_division=0)}

## Table 4.5: zero-shot transfer (Horizon-AGN)

In [ ]:
def eval_set(dom):
    d = DATA[dom]
    idx = np.arange(len(d["y"]))
    if d["sub"] is not None and (d["sub"] == "val").any():
        idx = np.where(d["sub"] == "val")[0]
    return idx[d["labeled"][idx]]           # labeled rows only (no-op for TNG/Horizon)

ZS = {}
for m in src["feat"]:
    print(f"=== {m} : zero-shot (probe trained on TNG) ===")
    print(f"  {'domain':8s} {'Acc':>6} {'AUC':>6} {'P':>6} {'R':>6} {'F1':>6} | supervised AUC band")
    for dom in [SOURCE] + TARGETS:
        if m not in DATA[dom]["feat"]:
            continue
        idx = eval_set(dom)
        X = SCAL[m].transform(np.asarray(DATA[dom]["feat"][m][idx], dtype=np.float32))
        p = PROBE[m].predict_proba(X)[:, 1]
        mt = metrics(DATA[dom]["y"][idx], p)
        ZS[(m, dom)] = mt
        band = SUP.get(dom, {}).get("auc", (np.nan, np.nan))
        print(f"  {dom:8s} {mt['acc']:6.3f} {mt['auc']:6.3f} {mt['P']:6.3f} {mt['R']:6.3f} {mt['F1']:6.3f} | {band[0]:.2f}-{band[1]:.2f}")

## HSC: faithful paper protocol (Sect. 5.3, Tables 4-5)

The paper never fits anything on HSC -- the TNG-trained models are applied **zero-shot**, and
the visual labels serve only as an evaluation reference. There is **no train/val split on HSC**;
what varies is the *negative class*, in three scenarios:

1. **Best case** (their Table 4): visual major mergers vs **clear non-mergers** (disturbed excluded)
2. **Realistic** (Table 5): negatives = non-mergers **+ disturbed**
3. **Worst case** (Table 5): negatives = **disturbed only**

Recall must be identical across the three rows of each model (TP and FN never change) --
that is the built-in correctness check. The first cell cross-tabs the catalog against the
paper's group counts (~1243 mergers / ~1095 clear non / ~868 disturbed); if those do not
reconstruct, resolve that before comparing numbers to their tables.

In [ ]:
d = DATA["hsc"]
mrg  = d["labeled"] & (d["y"] == 1)
dist = (d["dist"] == 1) if d["dist"] is not None else np.zeros(len(d["y"]), bool)
non  = d["labeled"] & (d["y"] == 0) & ~dist

print("labeled rows:", d["labeled"].sum())
print(f"visual major mergers : {mrg.sum():5,}   (paper: 1,243)")
print(f"clear non-mergers    : {non.sum():5,}   (paper: ~1,095)")
print(f"disturbed/minor      : {(d['labeled'] & dist).sum():5,}   (paper:   868)")
if d["dist"] is not None:
    print("\ncrosstab over labeled rows (is_major_merger x is_disturbed):")
    for ym in (0, 1):
        for dm in (0, 1):
            n = (d["labeled"] & (d["y"] == ym) & (dist == dm)).sum()
            print(f"  is_major_merger={ym}, is_disturbed={dm}: {n:,}")
else:
    print("\nWARNING: no is_disturbed field -> scenarios 2 and 3 reduce to scenario 1.")

## Tables 4.6 and 4.7: HSC scenarios

The best-case rows are Table 4.6.

In [ ]:
SCEN = {"best case (vs clear non)":       non,
        "realistic (vs non + disturbed)": non | (d["labeled"] & dist),
        "worst case (vs disturbed only)": d["labeled"] & dist}

SCEN_RESULTS = {}
for m in PROBE:
    print(f"--- {m} : zero-shot TNG probe on HSC visual sample ---")
    print(f"  {'scenario':32s} {'n':>6} {'Acc':>6} {'AUC':>6} {'P':>6} {'R':>6} {'F1':>6}")
    for sname, neg in SCEN.items():
        sel = np.where(mrg | neg)[0]
        if len(sel) == 0 or mrg[sel].sum() in (0, len(sel)):
            print(f"  {sname:32s}   skipped (one class)"); continue
        X = SCAL[m].transform(np.asarray(d["feat"][m][sel], dtype=np.float32))
        p = PROBE[m].predict_proba(X)[:, 1]
        mt = metrics(mrg[sel].astype(int), p)
        SCEN_RESULTS[(m, sname)] = mt
        print(f"  {sname:32s} {len(sel):6,} {mt['acc']:6.3f} {mt['auc']:6.3f} {mt['P']:6.3f} {mt['R']:6.3f} {mt['F1']:6.3f}")
    print("  (check: R identical down the column -- negatives never change TP or FN)\n")